In [1]:
import pandas as pd, sqlite3
from google.colab import files

In [2]:
up = files.upload()
fname = next(iter(up))
df = pd.concat(pd.read_excel(fname, sheet_name=None).values(), ignore_index=True)
print(df.shape)
df.head()

Saving online_retail_II.xlsx to online_retail_II.xlsx
(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df['Invoice'] = df['Invoice'].astype(str)
issues_log = {
    'raw rows': len(df),
    'exact duplicate rows': int(df.duplicated().sum()),
    'missing Customer ID': int(df['Customer ID'].isna().sum()),
    'cancellations (Invoice starts with C)': int(df['Invoice'].str.startswith('C').sum()),
    'quantity <= 0': int((df['Quantity'] <= 0).sum()),
    'price <= 0': int((df['Price'] <= 0).sum()),
}

df = df.drop_duplicates()
df = df.dropna(subset=['Customer ID'])
df = df[~df['Invoice'].str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
issues_log['clean rows'] = len(df)
issues_log['unique customers'] = int(df['Customer ID'].nunique())

print('df shape after cleaning:', df.shape)
print(issues_log)

df shape after cleaning: (779425, 8)
{'raw rows': 1067371, 'exact duplicate rows': 34335, 'missing Customer ID': 243007, 'cancellations (Invoice starts with C)': 19494, 'quantity <= 0': 22950, 'price <= 0': 6207, 'clean rows': 779425, 'unique customers': 5878}


In [4]:
before = len(df)
df = df[df['StockCode'].astype(str).str.match(r'^\d{5}')]
issues_log['non-product rows removed'] = before - len(df)
issues_log['clean rows (final)'] = len(df)
issues_log['unique customers (final)'] = int(df['Customer ID'].nunique())
print('removed:', before - len(df), '| rows now:', len(df), '| customers:', df['Customer ID'].nunique())

removed: 2848 | rows now: 776577 | customers: 5852


In [6]:
print('date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
print('missing values:\n', df.isna().sum())
print('min quantity:', df['Quantity'].min(), '| min price:', df['Price'].min())
print('max quantity:', df['Quantity'].max(), '| max price:', df['Price'].max())

date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
missing values:
 Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64
min quantity: 1 | min price: 0.03
max quantity: 80995 | max price: 649.5


In [7]:
df['Revenue'] = df['Quantity'] * df['Price']
print(df.nlargest(5, 'Revenue')[['Invoice','InvoiceDate','Description','Quantity','Price','Revenue','Customer ID']].to_string())

        Invoice         InvoiceDate                          Description  Quantity   Price   Revenue  Customer ID
1065882  581483 2011-12-09 09:15:00          PAPER CRAFT , LITTLE BIRDIE     80995    2.08  168469.6      16446.0
587080   541431 2011-01-18 10:01:00       MEDIUM CERAMIC TOP STORAGE JAR     74215    1.04   77183.6      12346.0
748132   556444 2011-06-10 15:28:00       PICNIC BASKET WICKER 60 PIECES        60  649.50   38970.0      15098.0
432176   530715 2010-11-04 11:36:00  ROTATING SILVER ANGELS T-LIGHT HLDR      9360    1.69   15818.4      15838.0
228042   511465 2010-06-08 12:59:00                  PINK PAPER PARASOL       3500    2.55    8925.0      18008.0


In [8]:
bulk = df[df['Quantity'] > 50000]['Invoice'].unique()
print('removing invoices:', bulk)
df = df[~df['Invoice'].isin(bulk)]
issues_log['extreme bulk orders removed (>50,000 units)'] = len(bulk)

removing invoices: ['541431' '581483']


In [9]:
raw = pd.read_excel(fname, sheet_name='Year 2010-2011')
print(raw[raw['Quantity'].isin([-80995, -74215])][['Invoice','InvoiceDate','Quantity','Customer ID']])

        Invoice         InvoiceDate  Quantity  Customer ID
61624   C541433 2011-01-18 10:17:00    -74215      12346.0
540422  C581484 2011-12-09 09:27:00    -80995      16446.0


In [10]:
df = df.rename(columns={'Customer ID': 'CustomerID'})
df['CustomerID'] = df['CustomerID'].astype(int)
df['Revenue'] = (df['Quantity'] * df['Price']).round(2)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate']).dt.strftime('%Y-%m-%d %H:%M:%S')
con = sqlite3.connect('retail.db')
df.to_sql('transactions', con, index=False, if_exists='replace')

776575

In [11]:
rfm_sql = """
WITH snap AS (
  SELECT date(MAX(InvoiceDate), '+1 day') AS d FROM transactions
),
rfm AS (
  SELECT CustomerID,
         CAST(julianday((SELECT d FROM snap)) - julianday(date(MAX(InvoiceDate))) AS INTEGER) AS Recency,
         COUNT(DISTINCT Invoice) AS Frequency,
         ROUND(SUM(Revenue), 2)  AS Monetary
  FROM transactions
  GROUP BY CustomerID
),
pct AS (
  SELECT *,
         CUME_DIST() OVER (ORDER BY Recency DESC) AS r_pct,
         CUME_DIST() OVER (ORDER BY Frequency)    AS f_pct,
         CUME_DIST() OVER (ORDER BY Monetary)     AS m_pct
  FROM rfm
),
scored AS (
  SELECT CustomerID, Recency, Frequency, Monetary,
    CASE WHEN r_pct <= 0.2 THEN 1 WHEN r_pct <= 0.4 THEN 2 WHEN r_pct <= 0.6 THEN 3 WHEN r_pct <= 0.8 THEN 4 ELSE 5 END AS R,
    CASE WHEN f_pct <= 0.2 THEN 1 WHEN f_pct <= 0.4 THEN 2 WHEN f_pct <= 0.6 THEN 3 WHEN f_pct <= 0.8 THEN 4 ELSE 5 END AS F,
    CASE WHEN m_pct <= 0.2 THEN 1 WHEN m_pct <= 0.4 THEN 2 WHEN m_pct <= 0.6 THEN 3 WHEN m_pct <= 0.8 THEN 4 ELSE 5 END AS M
  FROM pct
)
SELECT *,
  CASE
    WHEN R >= 4 AND F >= 4 THEN 'Champions'
    WHEN R >= 3 AND F >= 3 THEN 'Loyal Customers'
    WHEN R >= 4 AND F <= 2 THEN 'New & Promising'
    WHEN R = 3 AND F <= 2  THEN 'Need Attention'
    WHEN R <= 2 AND F >= 3 THEN 'At Risk'
    ELSE 'Lost'
  END AS Segment
FROM scored
"""
rfm = pd.read_sql(rfm_sql, con)
rfm.to_sql('rfm', con, index=False, if_exists='replace')
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R,F,M,Segment
0,17056,739,1,128.60,1,2,1,Lost
1,17592,739,1,148.30,1,2,1,Lost
2,14654,739,1,246.86,1,2,1,Lost
3,13526,739,2,1182.00,1,3,3,At Risk
4,15833,738,1,80.40,1,2,1,Lost


In [12]:
summary = pd.read_sql("""
SELECT Segment,
       COUNT(*) AS Customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM rfm), 1) AS PctCustomers,
       ROUND(AVG(Recency), 0) AS AvgRecencyDays,
       ROUND(AVG(Frequency), 1) AS AvgOrders,
       ROUND(AVG(Monetary), 0) AS AvgSpend,
       ROUND(SUM(Monetary), 0) AS TotalRevenue,
       ROUND(100.0 * SUM(Monetary) / (SELECT SUM(Monetary) FROM rfm), 1) AS PctRevenue
FROM rfm GROUP BY Segment ORDER BY TotalRevenue DESC""", con)
print(summary.to_string(index=False))

        Segment  Customers  PctCustomers  AvgRecencyDays  AvgOrders  AvgSpend  TotalRevenue  PctRevenue
      Champions       1583          27.1            21.0       14.7    7560.0    11968093.0        71.1
Loyal Customers       1444          24.7            78.0        4.7    1756.0     2535761.0        15.1
        At Risk       1206          20.6           380.0        4.1    1462.0     1763058.0        10.5
           Lost       1134          19.4           474.0        1.0     331.0      375432.0         2.2
 Need Attention        246           4.2           105.0        1.0     410.0      100891.0         0.6
New & Promising        239           4.1            30.0        1.0     333.0       79679.0         0.5


In [13]:
rfm.to_csv('rfm_customers.csv', index=False)
summary.to_csv('segment_summary.csv', index=False)
pd.Series(issues_log).to_csv('cleaning_log.csv', header=['count'])
from google.colab import files
for f in ['rfm_customers.csv', 'segment_summary.csv', 'cleaning_log.csv']:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>